In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# 12 — Environment provenance: where the compiled packages come from

**Fix.** The environment is now reproducible: Venti is pinned to a commit, Python is bounded to
`>=3.11,<3.14`, and `docker/conda-lock.txt` / `pixi.lock` were regenerated so that **every compiled
package, `netcdf4` included, comes from conda-forge and shares one HDF5**.

**Where the defect was** (paths at commit `d0901b3`, the state before the fix):

| Defect | Location |
|---|---|
| Venti dependency had no ref (`venti @ git+https://github.com/opera-adt/Venti`) | `pyproject.toml:45` |
| Python had no upper bound, the solver picked 3.14 | `environment.yml:5`, `pyproject.toml:161` |
| Lock pinned python 3.14.5 and **omitted `netcdf4`** although `environment.yml:17` lists it | `docker/conda-lock.txt` |
| so `pip install cal-disp[download]` pulled the PyPI `netCDF4` wheel with its own bundled HDF5 | `docker/Dockerfile:67-70` |
| an empty second lock file next to the real one | `conda-lock.txt` (0 bytes) |
| every image reported `cal_disp` version `1.0.0` | `docker/Dockerfile:67` |

**What is demonstrated.**

1. For the kernel this notebook runs in: which of netCDF4, h5py, GDAL, rasterio, numpy, pyproj, shapely
   were installed by conda and which by pip, read from the `INSTALLER` files and `conda-meta/`.
2. The HDF5 library version netCDF4 and h5py are each linked against. They must be the same.
3. The same two checks in an environment built from the regenerated lock (the "after").
4. The Venti version and the commit it is pinned to.
5. What provenance a product carries: the version attributes of the golden product.
6. What the old and the new lock file contain, and which version a Docker image reports.

The notebook reports whatever environment its kernel runs in. Executed in the pixi `dev` environment
(solved from the fixed `pyproject.toml`, python 3.13) sections 1–2 show the **fixed** stack. Executed
in an environment built the old way they show the **defect**: in the pre-fix `cal-disp-gamma`
environment (python 3.14.5, `netcdf4` installed by pip) this notebook measured netCDF4 from the PyPI
wheel on HDF5 1.14.6 next to h5py on HDF5 2.1.0. No network access is needed.

## 1. The running kernel: conda or pip?

In [2]:
import json
import subprocess
import sys

import pandas as pd

# The probe is a plain script so that the very same code can inspect the
# running kernel (exec) and another environment's interpreter (subprocess).
PROBE = r"""
import importlib.metadata as md, json, sys
from pathlib import Path

# import name -> (distribution name on PyPI, conda package name)
PACKAGES = {
    "netCDF4": ("netCDF4", "netcdf4"),
    "h5py": ("h5py", "h5py"),
    "osgeo.gdal": ("GDAL", "gdal"),
    "rasterio": ("rasterio", "rasterio"),
    "numpy": ("numpy", "numpy"),
    "pyproj": ("pyproj", "pyproj"),
    "shapely": ("shapely", "shapely"),
}
conda_meta = Path(sys.prefix) / "conda-meta"
rows = []
for import_name, (dist_name, conda_name) in PACKAGES.items():
    dist = md.distribution(dist_name)
    installer = (dist.read_text("INSTALLER") or "?").strip()
    # conda-meta/<name>-<version>-<build>.json exists only for conda packages
    record = None
    for f in conda_meta.glob(conda_name + "-*.json"):
        meta = json.loads(f.read_text())
        if meta["name"] == conda_name:
            record = meta
    # wheels repaired by auditwheel carry their shared libraries in <pkg>.libs/
    site = Path(dist.locate_file(""))
    bundled = sorted(
        p.name for d in site.glob("*.libs") if d.name.lower().startswith(conda_name)
        for p in d.iterdir() if p.name.startswith(("libhdf5", "libnetcdf"))
    )
    rows.append({
        "import": import_name,
        "version": dist.version,
        "INSTALLER": installer,
        "conda-meta": f"{record['build']} ({'conda-forge' if 'conda-forge' in record['channel'] else record['channel']})" if record else "-",
        "bundled libs (wheel)": ", ".join(bundled) or "-",
        "origin": "conda" if record and installer == "conda" else "pip (PyPI wheel)",
    })

import netCDF4, h5py
print(json.dumps({
    "python": sys.version.split()[0],
    "prefix": sys.prefix,
    "rows": rows,
    "hdf5": {"netCDF4": netCDF4.__hdf5libversion__, "h5py": h5py.version.hdf5_version},
    "venti": md.version("venti"),
    "venti_direct_url": json.loads(md.distribution("venti").read_text("direct_url.json") or "{}"),
}))
"""


def probe(python: str | Path = sys.executable) -> dict:
    """Run PROBE with the given interpreter and return its report."""
    out = subprocess.run(
        [str(python), "-W", "ignore", "-c", PROBE], capture_output=True, text=True, check=True
    )
    return json.loads(out.stdout.strip().splitlines()[-1])


def show(report: dict) -> pd.DataFrame:
    prefix = Path(report["prefix"])
    if prefix.is_relative_to(REPO):                 # e.g. the pixi environment
        prefix = Path("<repo>") / prefix.relative_to(REPO)
    print(f"python {report['python']}   prefix: {prefix}")
    return pd.DataFrame(report["rows"]).set_index("import")


def origin_verdict(report: dict) -> str:
    from_pip = [row["import"] for row in report["rows"] if row["origin"] != "conda"]
    if not from_pip:
        return "OK: every compiled package was installed by conda"
    return f"DEFECT: installed by pip, with their own bundled libraries: {', '.join(from_pip)}"


kernel = probe()
display(show(kernel))
print(origin_verdict(kernel))

python 3.13.15   prefix: <repo>/.pixi/envs/dev


,version,INSTALLER,conda-meta,bundled libs (wheel),origin
import,,,,,
netCDF4,1.7.4,conda,nompi_py311hd34d2e2_111 (conda-forge),-,conda
h5py,3.16.0,conda,nompi_py313hce4e9db_104 (conda-forge),-,conda
osgeo.gdal,3.13.3,conda,np2py313hb3192b2_1 (conda-forge),-,conda
rasterio,1.5.1,conda,py313ha8e1118_1 (conda-forge),-,conda
numpy,2.5.3,conda,py313hb5f73ae_0 (conda-forge),-,conda
pyproj,3.8.0,conda,py313hd11bf8e_1 (conda-forge),-,conda
shapely,2.1.2,conda,py313h14b1fe9_3 (conda-forge),-,conda


OK: every compiled package was installed by conda


`INSTALLER` is the file pip and conda each write into a package's `*.dist-info/` directory, and
`conda-meta/<name>-<version>-<build>.json` is conda's own record, present only for packages conda
installed. A package with `INSTALLER = pip` and no conda record is a PyPI wheel; for `netCDF4` that
wheel carries private copies of `libhdf5` and `libnetcdf`, which the last column would list.

## 2. HDF5 versions of netCDF4 and h5py in the kernel

In [3]:
def hdf5_verdict(report: dict) -> str:
    h = report["hdf5"]
    same = h["netCDF4"] == h["h5py"]
    return (
        f"netCDF4 -> HDF5 {h['netCDF4']}    h5py -> HDF5 {h['h5py']}    "
        + ("OK: one HDF5 library" if same else "MISMATCH: two different HDF5 libraries in one process")
    )


print(hdf5_verdict(kernel))

netCDF4 -> HDF5 2.2.0    h5py -> HDF5 2.2.0    OK: one HDF5 library


The two versions must be equal. Two HDF5 libraries loaded into one process is the hazard the comment
in `environment.yml` describes: cal-disp reads products with netCDF4 (`workflow._read_wavelength_m`,
xarray's `netcdf4` engine) while h5py/h5netcdf and GDAL use the conda HDF5. Each library has its own
global state and its own lock, so reads from several threads can crash (Venti D23). An image built
from the old lock contained exactly that mismatch.

## 3. The same checks in an environment built from the regenerated lock

`CAL_DISP_LOCK_ENV` points to an environment created from `docker/conda-lock.txt` the way the
Dockerfile does it, followed by the pip install of the package:

```bash
mamba create -p $TMPDIR/cal-disp-lock/verifyenv --file docker/conda-lock.txt
$TMPDIR/cal-disp-lock/verifyenv/bin/python -m pip install -e ".[download,test]"
```

If that environment does not exist the cell says so and the notebook continues.

In [4]:
LOCK_ENV = Path(
    os.environ.get(
        "CAL_DISP_LOCK_ENV",
        Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal-disp-lock" / "verifyenv",
    )
)
lock_python = LOCK_ENV / "bin" / "python"
locked = probe(lock_python) if lock_python.exists() else None
show(locked) if locked else print(f"No environment at {LOCK_ENV}; create it as shown above.")

python 3.13.15   prefix: /u/aurora-r0/govorcin/tmp/cal-disp-lock/verifyenv


,version,INSTALLER,conda-meta,bundled libs (wheel),origin
import,,,,,
netCDF4,1.7.4,conda,nompi_py311hd34d2e2_111 (conda-forge),-,conda
h5py,3.16.0,conda,nompi_py313hce4e9db_104 (conda-forge),-,conda
osgeo.gdal,3.13.3,conda,np2py313hb3192b2_1 (conda-forge),-,conda
rasterio,1.5.1,conda,py313ha8e1118_1 (conda-forge),-,conda
numpy,2.5.3,conda,py313hb5f73ae_0 (conda-forge),-,conda
pyproj,3.8.0,conda,py313hd11bf8e_1 (conda-forge),-,conda
shapely,2.1.2,conda,py313h14b1fe9_3 (conda-forge),-,conda


In [5]:
if locked:
    print(hdf5_verdict(locked))
    assert locked["hdf5"]["netCDF4"] == locked["hdf5"]["h5py"], "HDF5 mismatch in the locked env"
    assert all(row["origin"] == "conda" for row in locked["rows"]), "a compiled package came from pip"
    assert locked["python"].startswith("3.13."), locked["python"]

    before = {r["import"]: r for r in kernel["rows"]}["netCDF4"]
    after = {r["import"]: r for r in locked["rows"]}["netCDF4"]
    display(
        pd.DataFrame(
            {
                "kernel": [kernel["python"], before["origin"], kernel["hdf5"]["netCDF4"], kernel["hdf5"]["h5py"]],
                "env from the new lock": [locked["python"], after["origin"], locked["hdf5"]["netCDF4"], locked["hdf5"]["h5py"]],
            },
            index=["python", "netCDF4 origin", "HDF5 of netCDF4", "HDF5 of h5py"],
        )
    )

netCDF4 -> HDF5 2.2.0    h5py -> HDF5 2.2.0    OK: one HDF5 library


,kernel,env from the new lock
python,3.13.15,3.13.15
netCDF4 origin,conda,conda
HDF5 of netCDF4,2.2.0,2.2.0
HDF5 of h5py,2.2.0,2.2.0


## 4. Venti: installed version and pinned commit

In [6]:
import tomllib

pyproject = tomllib.loads((REPO / "pyproject.toml").read_text())
venti_req = next(d for d in pyproject["project"]["dependencies"] if d.startswith("venti"))
pinned_commit = venti_req.rsplit("@", 1)[1] if venti_req.count("@") == 2 else None

print("pyproject.toml  :", venti_req)
print("pinned commit   :", pinned_commit)
print("python bound    :", pyproject["tool"]["pixi"]["dependencies"]["python"], "(pixi),",
      next(l.strip() for l in (REPO / "environment.yml").read_text().splitlines() if "python>=" in l), "(environment.yml)")
print()
for label, report in [("kernel", kernel), ("locked env", locked)]:
    if report:
        commit = report["venti_direct_url"].get("vcs_info", {}).get("commit_id")
        print(f"{label:11s}: venti {report['venti']}   installed from commit {commit}")
        # setuptools_scm puts the abbreviated commit after "+g"
        assert report["venti"].split("+g")[1] == pinned_commit[:9]
        assert commit == pinned_commit

pyproject.toml  : venti @ git+https://github.com/opera-adt/Venti@2a7e61f0ce3c51658afd4bbd9bf8570c72f4ac4e
pinned commit   : 2a7e61f0ce3c51658afd4bbd9bf8570c72f4ac4e
python bound    : >=3.11,<3.14 (pixi), - python>=3.11,<3.14 (environment.yml)

kernel     : venti 0.0.post1.dev80+g2a7e61f0c   installed from commit 2a7e61f0ce3c51658afd4bbd9bf8570c72f4ac4e
locked env : venti 0.0.post1.dev80+g2a7e61f0c   installed from commit 2a7e61f0ce3c51658afd4bbd9bf8570c72f4ac4e


pip records the exact commit a `git+https` requirement resolved to in `direct_url.json`; both
environments have the pinned commit. Before the fix the requirement had no ref, so the same
`pip install` on another day would have taken whatever Venti's default branch pointed to, with no
change visible in this repository.

## 5. The provenance a product carries

In [7]:
import h5py

golden_files = sorted((GOLDEN / "golden_output").glob("*.nc"))
if not golden_files:
    print(f"No golden product under {GOLDEN / 'golden_output'}; set CAL_DISP_GOLDEN_DIR.")
else:
    golden = golden_files[0]
    print(golden.name)

    ABSENT = "(absent)"

    def text(value):
        value = value[()] if hasattr(value, "shape") else value
        return value.decode() if isinstance(value, bytes) else str(value)

    # The software versions are stored once, as datasets of the /metadata group.
    DATASETS = [
        "metadata/cal_disp_software_version",
        "metadata/venti_software_version",
        "metadata/source_data_software_disp_version",
        "identification/product_version",
    ]
    with h5py.File(golden) as f:
        provenance = {f"/{path}": text(f[path]) if path in f else ABSENT for path in DATASETS}
        # Earlier products repeated the version in root attributes; report them if present.
        root_attrs = {name: text(f.attrs[name]) for name in ("software", "software_version") if name in f.attrs}
    display(pd.Series(provenance, name="golden product").to_frame())

    missing = [path for path, value in provenance.items() if value == ABSENT]
    if missing:
        print("Note: not in this product:", ", ".join(missing))
    if root_attrs:
        print("Root attributes (older product layout):", root_attrs)
    else:
        print("No root attributes `software`/`software_version`: the version is recorded only in /metadata.")

    venti_version = provenance["/metadata/venti_software_version"]
    if venti_version == ABSENT:
        print("Note: no Venti version recorded, cannot compare it with the pin.")
    elif venti_version.split("+g")[-1] == pinned_commit[:9]:
        print(f"The golden product was made with the pinned Venti commit ({pinned_commit[:9]}).")
    else:
        print(f"Note: the golden product was made with Venti {venti_version}, not the pinned commit {pinned_commit[:9]}.")

OPERA_L4_DISP-CAL-S1_IW_F08882_VV_20220111T002651Z_20220722T002657Z_v1.0_20260930T213458Z.nc


,golden product
/metadata/cal_disp_software_version,0.2.post1.dev62+g4117f9341.d20260930
/metadata/venti_software_version,0.0.post1.dev80+g2a7e61f0c
/metadata/source_data_software_disp_version,1.0
/identification/product_version,1.0


No root attributes `software`/`software_version`: the version is recorded only in /metadata.
The golden product was made with the pinned Venti commit (2a7e61f0c).


The product records the cal-disp and Venti versions in `/metadata` as `setuptools_scm` strings, i.e.
tag, distance and commit (`<tag>.post1.dev<N>+g<commit>`). That is only as good as the version the installed
package reports, which is why the hard-coded `1.0.0` of the Docker image mattered (next section):
every product made in any image would have said `cal_disp_software_version = 1.0.0`.

The product does not record the versions of the compiled libraries (HDF5, GDAL, numpy); those are
fixed by the lock file, so the lock is part of a product's provenance.

## 6. What the lock files contain, and the version an image reports

In [8]:
import re

BEFORE = "d0901b3"   # the commit before this fix


def git_show(path: str) -> str | None:
    """File content at the commit before the fix (None outside a git checkout)."""
    try:
        return subprocess.run(
            ["git", "show", f"{BEFORE}:{path}"], cwd=REPO, capture_output=True, text=True, check=True
        ).stdout
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def lock_summary(text: str) -> dict:
    """Version-build of the packages that matter, from an @EXPLICIT lock."""
    found = {}
    for name in ["python", "netcdf4", "libnetcdf", "hdf5", "h5py", "numpy"]:
        m = re.search(rf"/{name}-(\d[^/#]*)\.(?:conda|tar\.bz2)#", text)
        found[name] = m.group(1) if m else "MISSING"
    found["packages"] = sum(line.startswith("https://") for line in text.splitlines())
    return found


old_lock = git_show("docker/conda-lock.txt")
new_lock = (REPO / "docker" / "conda-lock.txt").read_text()
table = {"new docker/conda-lock.txt": lock_summary(new_lock)}
if old_lock is not None:
    table = {f"old docker/conda-lock.txt ({BEFORE})": lock_summary(old_lock), **table}
display(pd.DataFrame(table))

new = lock_summary(new_lock)
assert new["python"].startswith("3.13.") and "MISSING" not in new.values()
root_lock = REPO / "conda-lock.txt"
print("root conda-lock.txt:", f"{root_lock.stat().st_size} bytes" if root_lock.exists() else "removed (it was an empty file)")

,old docker/conda-lock.txt (d0901b3),new docker/conda-lock.txt
python,3.14.5-habeac84_100_cp314,3.13.15-hf47f18c_103_cp313
netcdf4,MISSING,1.7.4-nompi_py311hd34d2e2_111
libnetcdf,4.10.0-nompi_h3c9b436_104,4.10.1-nompi_he3e3c8e_201
hdf5,2.1.0-nompi_hee75f7e_106,2.2.0-nompi_hc529623_100
h5py,3.16.0-nompi_py314hddf7a69_102,3.16.0-nompi_py313hce4e9db_104
numpy,2.4.6-py314h2b28147_0,2.5.3-py313hb5f73ae_0
packages,189,213


root conda-lock.txt: removed (it was an empty file)


In [9]:
old_dockerfile = git_show("docker/Dockerfile")
if old_dockerfile is not None:
    print("old Dockerfile :", next(l for l in old_dockerfile.splitlines() if "PRETEND_VERSION" in l).strip())
print("new Dockerfile :", *[l.strip() for l in (REPO / "docker" / "Dockerfile").read_text().splitlines()
                           if "CAL_DISP_VERSION" in l and not l.startswith("#")], sep="\n                 ")

helper = REPO / "docker" / "pep440-version.sh"
for describe in ["v0.2", "v0.2-27-g88c0c46", "88c0c46"]:
    version = subprocess.run([helper, describe], capture_output=True, text=True, check=True).stdout.strip()
    print(f"  git describe {describe:18s} -> CAL_DISP_VERSION={version}")

old Dockerfile : ENV SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CAL_DISP=1.0.0
new Dockerfile :
                 ARG CAL_DISP_VERSION=0.0.0.dev0+unknown
                 ENV SETUPTOOLS_SCM_PRETEND_VERSION_FOR_CAL_DISP=${CAL_DISP_VERSION}
  git describe v0.2               -> CAL_DISP_VERSION=0.2
  git describe v0.2-27-g88c0c46   -> CAL_DISP_VERSION=0.2.post1.dev27+g88c0c46
  git describe 88c0c46            -> CAL_DISP_VERSION=0.0.0.post1.dev0+g88c0c46


## Summary

**What the stale lock did.** `docker/conda-lock.txt` was generated before `netcdf4` was added to
`environment.yml` and never regenerated. The Dockerfile installs the conda layer from the lock and
then runs `pip install cal-disp[download]`; `netcdf4` is a dependency in `pyproject.toml`, the lock
did not provide it, so pip satisfied it with the PyPI wheel. That wheel bundles its own HDF5, a
different version from the conda HDF5 that h5py and GDAL use (measured in the pre-fix
`cal-disp-gamma` environment: HDF5 1.14.6 for netCDF4, 2.1.0 for h5py). The image therefore ran
with two HDF5 libraries in one process — the crash hazard `environment.yml` warns about — although
`environment.yml` itself was correct. The lock also pinned python 3.14.5, where the conda-forge
`netcdf4` build warns about the numpy ABI on import.

**After the fix** (sections 1–3): the pixi environment and an environment built from the regenerated
lock have python 3.13, `netcdf4` from conda-forge, and one HDF5 for netCDF4 and h5py; Venti is
installed from the pinned commit (section 4), and section 5 compares it with the Venti version the
golden product records.

**How to keep it that way.** After any change to `environment.yml`, regenerate the lock with
`docker/create-lockfile.sh --file environment.yml [--no-docker] > docker/conda-lock.txt` and check
that it contains `netcdf4`, `libnetcdf`, `hdf5` and `h5py` (section 6); after a change to the pixi
tables, `pixi install`.